In [1]:
import numpy as np

In [1]:
import pandas as pd
labels = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

df = pd.read_csv('SEP-28k-Filtered_with_Split_and_Path.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28177 entries, 0 to 28176
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Unnamed: 0        28177 non-null  int64
 1   Show              28177 non-null  str  
 2   EpId              28177 non-null  int64
 3   ClipId            28177 non-null  int64
 4   SEP28k-T          28177 non-null  str  
 5   SEP28k-D          28177 non-null  str  
 6   Prolongation      28177 non-null  int64
 7   Block             28177 non-null  int64
 8   SoundRep          28177 non-null  int64
 9   WordRep           28177 non-null  int64
 10  Interjection      28177 non-null  int64
 11  NoStutteredWords  28177 non-null  int64
 12  filepath          28177 non-null  str  
dtypes: int64(9), str(4)
memory usage: 2.8 MB


In [3]:

value_counts_table = (
    pd.concat(
        [df[col].value_counts().rename(col) for col in labels],
        axis=1
    )
)

value_counts_table

,Prolongation,Block,SoundRep,WordRep,Interjection
0,19631,16207,22563,23556,18519
1,5734,8600,3272,1851,3685
2,2022,2842,1479,1160,2595
3,790,528,863,1610,3378


In [4]:
two_plus = value_counts_table.loc[[2, 3]].sum()
two_plus

Prolongation    2812
Block           3370
SoundRep        2342
WordRep         2770
Interjection    5973
dtype: int64

In [5]:
df_filtered = df[
    df[labels].ge(2).any(axis=1) | df[labels].eq(0).all(axis=1)
].copy() #keep rows with >= 2, and rows with 0 (for fluent)


In [6]:
df_filtered.head()

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath
0,0,HeStutters,0,0,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_0.wav
1,1,HeStutters,0,1,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_1.wav
2,2,HeStutters,0,2,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_2.wav
4,4,HeStutters,0,4,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_4.wav
6,6,HeStutters,0,6,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_6.wav


In [7]:

value_counts_table = (
    pd.concat(
        [df_filtered[col].value_counts().rename(col) for col in labels],
        axis=1
    )
)

print(value_counts_table)
two_plus = value_counts_table.loc[[2, 3]].sum()
two_plus

   Prolongation  Block  SoundRep  WordRep  Interjection
0         14557  12790     15778    16207         12340
1          2657   3866      1906     1049          1713
2          2022   2842      1479     1160          2595
3           790    528       863     1610          3378


Prolongation    2812
Block           3370
SoundRep        2342
WordRep         2770
Interjection    5973
dtype: int64

In [8]:
#turn labels to (1,0) ground truths
df_filtered[labels] = (df_filtered[labels] >= 2).astype(np.float32)

In [9]:

df_filtered[labels]

,Prolongation,Block,SoundRep,WordRep,Interjection
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
28172,0.0,0.0,0.0,0.0,1.0
28173,0.0,0.0,1.0,0.0,0.0
28174,0.0,0.0,0.0,0.0,1.0
28175,0.0,0.0,0.0,0.0,1.0


In [10]:
df_filtered["fluent"] = df[labels].eq(0).all(axis=1).astype(int) #add a fluent flag if all disfluencies are 0.

In [11]:
sample_labels = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection', 'fluent']

In [12]:
value_counts_table = (
    pd.concat(
        [df_filtered[col].value_counts().rename(col) for col in sample_labels],
        axis=1
    )
)

print(value_counts_table)


     Prolongation  Block  SoundRep  WordRep  Interjection  fluent
0.0         17214  16656     17684    17256         14053   14022
1.0          2812   3370      2342     2770          5973    6004


In [13]:
#now if we want to upsample, we need to make entries with ONLY that specific label to be upsampled
#similarly for down sample, we need to make entries with ONLY that label to be downsampled. 


In [ ]:
#down sample to 1k each for better training.

In [13]:
import numpy as np

label_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection', 'fluent']
max_per_label = 1000

df_shuffled = df_filtered.sample(frac=1, random_state=42).reset_index(drop=True)

label_counts = {label: 0 for label in label_cols}
selected_indices = []

for idx, row in df_shuffled.iterrows():
    row_labels = [label for label in label_cols if row[label] == 1]
    
    # Check if adding this row would exceed any label limit
    if any(label_counts[label] >= max_per_label for label in row_labels):
        continue
    
    selected_indices.append(idx)
    
    for label in row_labels:
        label_counts[label] += 1
    
    # Stop if all labels reached limit
    if all(count >= max_per_label for count in label_counts.values()):
        break

df_downsampled = df_shuffled.loc[selected_indices].reset_index(drop=True)

In [ ]:
# #for now we can just downsample interjection and fluent to about 3k
# df_fluent = df_filtered[df_filtered['fluent'] == 1]
# df_other = df_filtered[df_filtered['fluent'] != 1]

# df_label_down = df_fluent.sample(n=3000, random_state=20)

# df_downsampled_fluent = pd.concat([df_label_down, df_other], ignore_index=True)

In [ ]:
# df_inter = df_downsampled_fluent[df_downsampled_fluent['Interjection'] == 1]
# df_other = df_downsampled_fluent[df_downsampled_fluent['Interjection'] != 1]

# df_label_down = df_inter.sample(n=3000, random_state=20)

# df_downsampled = pd.concat([df_label_down, df_other], ignore_index=True)

In [14]:
value_counts_table = (
    pd.concat(
        [df_downsampled[col].value_counts().rename(col) for col in sample_labels],
        axis=1
    )
)

print(value_counts_table)

     Prolongation  Block  SoundRep  WordRep  Interjection  fluent
0.0          4242   4242      4242     4242          4242    4242
1.0          1000   1000      1000     1000          1000    1000


In [17]:
#At this point, it's going it be a multilabel classfication problem. Each input can have multiple ground truths that are 1 for the 5 disfluencies. 
#If the 5 labels are all 0, that means that it's fluent.

In [15]:
df = df_downsampled.copy() 
df

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath,fluent
0,8750,StrongVoices,19,6,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/19/StrongVoices_19_6...,1
1,9682,StrongVoices,32,28,train,dev,1.0,0.0,0.0,1.0,0.0,0,SEP-28k_CLIP/StrongVoices/32/StrongVoices_32_2...,0
2,8274,StrongVoices,10,70,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/10/StrongVoices_10_7...,1
3,5027,IStutterSoWhat,3,68,dev,train,0.0,0.0,0.0,0.0,1.0,2,SEP-28k_CLIP/IStutterSoWhat/3/IStutterSoWhat_3...,0
4,2564,HeStutters,18,184,test,test,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/HeStutters/18/HeStutters_18_184.wav,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5237,2912,HeStutters,20,132,test,test,0.0,0.0,1.0,0.0,0.0,1,SEP-28k_CLIP/HeStutters/20/HeStutters_20_132.wav,0
5238,3905,HVSA,1,84,dev,train,0.0,0.0,1.0,0.0,0.0,1,SEP-28k_CLIP/HVSA/1/HVSA_1_84.wav,0
5239,22928,WomenWhoStutter,43,14,test,test,0.0,0.0,1.0,0.0,0.0,0,SEP-28k_CLIP/WomenWhoStutter/43/WomenWhoStutte...,0
5240,3755,HVSA,0,71,train,dev,0.0,0.0,1.0,0.0,0.0,0,SEP-28k_CLIP/HVSA/0/HVSA_0_71.wav,0


In [16]:
df['audio_path'] = (
    "../Training/" + 
    df['filepath']
)

In [17]:
df.head()

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath,fluent,audio_path
0,8750,StrongVoices,19,6,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/19/StrongVoices_19_6...,1,../Training/SEP-28k_CLIP/StrongVoices/19/Stron...
1,9682,StrongVoices,32,28,train,dev,1.0,0.0,0.0,1.0,0.0,0,SEP-28k_CLIP/StrongVoices/32/StrongVoices_32_2...,0,../Training/SEP-28k_CLIP/StrongVoices/32/Stron...
2,8274,StrongVoices,10,70,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/10/StrongVoices_10_7...,1,../Training/SEP-28k_CLIP/StrongVoices/10/Stron...
3,5027,IStutterSoWhat,3,68,dev,train,0.0,0.0,0.0,0.0,1.0,2,SEP-28k_CLIP/IStutterSoWhat/3/IStutterSoWhat_3...,0,../Training/SEP-28k_CLIP/IStutterSoWhat/3/IStu...
4,2564,HeStutters,18,184,test,test,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/HeStutters/18/HeStutters_18_184.wav,1,../Training/SEP-28k_CLIP/HeStutters/18/HeStutt...


In [18]:
import os
test_path = df['audio_path'].iloc[0]
print(f"🔍 Checking first file at: {os.path.abspath(test_path)}")
if not os.path.exists(test_path):
    print("❌ ERROR: File STILL not found! Please stop and double-check your folder structure.")
else:
    print("✅ File found! Pathing is correct.\n")

🔍 Checking first file at: c:\MyStuff\SJSU\Spring 2026\Ling 230\DisfluencyProject\disfluency_detection\Training\SEP-28k_CLIP\StrongVoices\19\StrongVoices_19_6.wav
✅ File found! Pathing is correct.



In [19]:
import pandas as pd
import torch
import librosa
import os
from transformers import WhisperFeatureExtractor, WhisperModel
from tqdm.notebook import tqdm # visualization
import numpy as np 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader # for training
from sklearn.utils.class_weight import compute_class_weight # class weights
from sklearn.model_selection import train_test_split # split data
from sklearn.preprocessing import LabelEncoder #transform labels
import matplotlib.pyplot as plt # visualization
import seaborn as sns # visualization
from sklearn.metrics import confusion_matrix, classification_report # visualization
import copy # copy and save model 
import math
import gc

In [20]:
# Set up the Whisper Model
model_name = "openai/whisper-base"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name)

# find Mac GPU, also allowed for NVIDIA GPU
if torch.backends.mps.is_available():
    device = "mps"
    print("Using Mac GPU (MPS) for acceleration!")
elif torch.cuda.is_available():
    device = "cuda"
    print("Using NVIDIA GPU (CUDA) for acceleration!")
else:
    device = "cpu"
    print("Using CPU. (No GPU detected)")

model.to(device)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using NVIDIA GPU (CUDA) for acceleration!


WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 512)
    (layers): ModuleList(
      (0-5): 6 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=False)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=512, out_features=2048, bias=True)
        (fc2): Linear(in_features=2048, out_features=512, bias=True)
        (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )


In [21]:
def get_whisper_embedding_sequence(audio_path):
    if not os.path.exists(audio_path): return None
    try:
        # Load audio at 16k Hz
        speech_array, sampling_rate = librosa.load(audio_path, sr=16000)
        inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)
        
        with torch.no_grad():
            encoder_outputs = model.encoder(input_features)
            
        # remove squeeze dimension
        embedding = encoder_outputs.last_hidden_state.squeeze(0).cpu().numpy()
        return embedding 
    except Exception as e:
        print(f"Error: {e}")
        return None

In [22]:
# Process the files
print(f"Processing audio files into sequences...")
all_embeddings = []
valid_indices = []

for idx, path in tqdm(enumerate(df['audio_path']), total=len(df)):
    emb = get_whisper_embedding_sequence(path) 
    
    if emb is not None:
        all_embeddings.append(emb)
        valid_indices.append(idx)

# Save as a true 3D Numpy Array
# (Samples, 150, 512)
np.save('whisper_embeddings.npy', np.array(all_embeddings))


clean_df = df.iloc[valid_indices].copy()
clean_df.to_csv('SEP-28k_with_paths.csv', index=False)
print(f"🎉 Finished! Shape saved: {np.array(all_embeddings).shape}")

Processing audio files into sequences...


  0%|          | 0/5242 [00:00<?, ?it/s]

🎉 Finished! Shape saved: (5242, 1500, 512)


MODEL

In [ ]:
label_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

In [ ]:
label_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']
print("Loading data...")
X_raw = np.load('whisper_embeddings.npy') 

# Slice to 3 seconds (150 tokens) to remove 1,350 tokens of silence
# "memory footprint" from 12GB to ~1.2GB I had AI help me with this
X = X_raw[:, :150, :] 

# Free up the old memory immediately
print(f"Original Shape: {X_raw.shape} | Optimized Shape: {X.shape}")
del X_raw
gc.collect() 

df = pd.read_csv('SEP-28k_with_paths.csv')
y_text = df[label_cols]

label_encoder = LabelEncoder()
y = df[label_cols].to_numpy(dtype=np.float32)
num_classes = 5

# Split off 20% for Test, then 25% of 80% for Val
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Convert to Tensors (Directly onto CPU first, then DataLoaders handle device move)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32, shuffle=False)

print(f"🎉 Memory Cleaned. Data Splits -> Train: {len(X_train)} | Val: {len(X_val)}")


In [ ]:
# Cnn + transformer

class TemporalCNNEncoder(nn.Module):
    def __init__(self):
        super(TemporalCNNEncoder, self).__init__()
        # Whisper base dim is 512. We treat these as "Channels" for the CNN
        self.conv1 = nn.Conv1d(in_channels=512, out_channels=256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(256)
        self.conv2 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu = nn.GELU()
        self.pool = nn.MaxPool1d(kernel_size=2) 

    def forward(self, x):
        # x enters as [Batch, 150, 512]. Conv1d: [Batch, Channels, Time]
        x = x.transpose(1, 2) 
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        return x.transpose(1, 2) # Returns [Batch, 75, 128] for Transformer
    

class TransformerBrain(nn.Module):
    def __init__(self, num_classes, d_model=128, seq_length=75):
        super(TransformerBrain, self).__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_length, d_model) * 0.01)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=512, dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = x + self.pos_embedding
        x = self.transformer(x)
        x = x.mean(dim=1) 
        return self.fc(x)
    

class EndToEndStutterModel(nn.Module):
    def __init__(self, num_classes):
        super(EndToEndStutterModel, self).__init__()
        self.cnn = TemporalCNNEncoder()
        self.transformer = TransformerBrain(num_classes)

    def forward(self, x):
        return self.transformer(self.cnn(x))

In [ ]:
# training and weights 
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

device = torch.device(device)
print("Using device:", device)

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

model = EndToEndStutterModel(num_classes).to(device)
criterion = nn.BCEWithLogitsLoss() #for multilabel classification
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

epochs = 100
patience = 15
best_val_loss = float('inf')
patience_counter = 0
best_model_weights = copy.deepcopy(model.state_dict())


In [ ]:
# training 
print(f"Starting training on {device}...")

for epoch in range(epochs):
    model.train()
    running_train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
        
    model.eval()
    running_val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            running_val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += batch_y.size(0)
            correct_val += (predicted == batch_y).sum().item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_acc = 100 * correct_val / total_val
    print(f"Epoch {epoch+1:02d} | Train Loss: {running_train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping triggered at Epoch {epoch+1}!")
            model.load_state_dict(best_model_weights)
            break
 
print("\n🎉 Training complete!")

In [ ]:


#saving model
MODEL_PATH = "downsampled_model.pt"

torch.save(model.state_dict(), MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")



RUN TEST OF MODEL

In [ ]:
from sklearn.metrics import f1_score

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoader
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)

# Evaluate
model.eval()
all_preds = []
all_labels = []
all_probs = []

print("Evaluating on the completely unseen Test Set...")

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        
        probs = torch.sigmoid(outputs)
        all_probs.append(probs.cpu())
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

f1_micro = f1_score(all_labels, all_preds, average="micro")
f1_macro = f1_score(all_labels, all_preds, average="macro")
all_probs = torch.cat(all_probs, dim=0)
probs_np = all_probs.numpy()

print("Micro F1:", f1_micro)
print("Macro F1:", f1_macro)



In [ ]:
def clipName (row):
    show = str(row["Show"])
    episode = str(row["EpId"])
    clip = str(row["ClipId"])

    return f"{show}_{episode}_{clip}"

In [ ]:
probs_df = pd.DataFrame(probs_np, columns=labels)

In [ ]:
clips = df.apply(clipName, axis=1)
probs_df.insert(0, "Clip", clips.values)

